<a href="https://colab.research.google.com/github/sathundorn/Super-AI-Engineer-Season-6/blob/main/601402_mini_Hackathon_Form_Data_to_Insight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install squarify

In [ ]:
# นำเข้าไลบรารีที่จำเป็นสำหรับการวิเคราะห์ข้อมูล
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import squarify
import ipywidgets as widgets
from ipywidgets import interact
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
mpl.font_manager.fontManager.addfont('thsarabunnew-webfont.ttf')
plt.rc('font', family='TH Sarabun New', size=14)

# 1️⃣ Data Preparation

โหลดข้อมูล Open Data เกี่ยวกับ สถิติการเกิดอัคคีภัยในประเทศไทย ปีพ.ศ. 2562 พร้อมข้อมูลความเสียหาย เพื่อนำมาวิเคราะห์แนวโน้มและความเสียหายจากเหตุไฟไหม้

In [ ]:
# โหลดชุดข้อมูลจากไฟล์ CSV เข้าสู่ DataFrame ของ pandas
df = pd.read_csv('/content/gd026_fire_stat2562_final_with_province_en.csv')

# ตรวจสอบโครงสร้างข้อมูล

In [ ]:
# แสดงข้อมูลเชิงสรุปของ DataFrame เพื่อตรวจสอบประเภทข้อมูลและค่าที่ไม่ใช่ Null
df.info()

# ตรวจสอบขนาดของข้อมูล

In [ ]:
# แสดงจำนวนแถวของข้อมูลใน DataFrame
print(df.shape[0]) #ดูแถวของข้อมูล

In [ ]:
# แสดงจำนวนคอลัมน์ของข้อมูลใน DataFrame
print(df.shape[1]) #ดูคอลัมน์ของข้อมูล

In [ ]:
# แสดง 5 แถวแรกของ DataFrame เพื่อดูตัวอย่างข้อมูลเบื้องต้น
df.head(5)

In [ ]:
# แสดง 5 แถวสุดท้ายของ DataFrame เพื่อดูตัวอย่างข้อมูลส่วนท้าย
df.tail()

# Insight จากการตรวจสอบพบว่า

1.dataset มี 72 rows และ 35 columns

2.แถวแรกของข้อมูลจริงเป็น header ซ้ำ

3.มีแถวสรุปรวมและข้อความที่ไม่เหมาะกับการนำมา Analysis

# ลบแถว header ที่ซ้ำ

In [ ]:
# ลบแถวแรกของ DataFrame ที่เป็น header ซ้ำซ้อนทิ้งไป
df = df.iloc[1:].copy()

 # ลบแถวสรุปรวมและแถวข้อความท้ายไฟล์

In [ ]:
# ลบแถวข้อมูลสรุปและข้อความท้ายไฟล์ โดยคงไว้เฉพาะ 65 แถวแรกที่เป็นข้อมูลจังหวัดที่ใช้งานได้จริง
df = df.iloc[:65].copy()

# ตรวจสอบขนาดของข้อมูลหลังจากการลบ header ที่ซ้ำ และ แถวสรุปรวมและแถวข้อความท้ายไฟล์

In [ ]:
# แสดงจำนวนแถวของข้อมูลใน DataFrame หลังจากการลบแถวที่ไม่ต้องการ
print(df.shape[0]) #ดูแถวของข้อมูล

In [ ]:
# แสดงจำนวนคอลัมน์ของข้อมูลใน DataFrame หลังจากการลบแถวที่ไม่ต้องการ (จำนวนคอลัมน์ไม่ควรเปลี่ยนแปลง)
print(df.shape[1]) #ดูคอลัมน์ของข้อมูล

# จัดการ Missing Values

In [ ]:
# แทนที่เครื่องหมาย '-' ด้วยค่า NaN (Not a Number) เพื่อให้สามารถจัดการค่าว่างได้ง่ายขึ้น
df = df.replace('-', np.nan)
df.head()

ใน dataset ใช้เครื่องหมาย - แทนค่าที่ไม่มีข้อมูล
จึงแปลงเป็น NaN เพื่อให้สามารถวิเคราะห์ข้อมูลได้ถูกต้อง

# แปลงข้อมูลเป็นตัวเลข

In [ ]:
# วนลูปผ่านคอลัมน์ทั้งหมด (ยกเว้นคอลัมน์แรก 'province' และคอลัมน์สุดท้าย 'province_en')
for col in df.columns[1:34]:
    # แปลงคอลัมน์ให้เป็น string และลบเครื่องหมายจุลภาค (',') ออก
    df[col] = df[col].astype(str).str.replace(',', '')
    # แปลงคอลัมน์เป็นตัวเลข (numeric) หากแปลงไม่ได้ให้เป็น NaN (coerce)
    df[col] = pd.to_numeric(df[col], errors='coerce')
# แสดงข้อมูลเชิงสรุปของ DataFrame อีกครั้งเพื่อตรวจสอบประเภทข้อมูลที่เปลี่ยนไป
df.info()

ปัญหาที่พบในข้อมูล

-ตัวเลขมี comma เช่น 6,738,000

-pandas อ่านเป็น string

-จึงต้อง
ลบ comma
แปลงเป็น numeric

# ตรวจสอบ Missing Values

In [ ]:
# หาค่าว่างในแต่ละคอลัมน์และแสดงผลรวม
print("--- Missing Values Count ---")
print(df.isnull().sum())

# ตัวอย่างที่พบ

department_stores ไม่มีข้อมูล

firefighter_fatalities มีข้อมูลน้อยมาก

คอลัมน์เหล่านี้อาจไม่เหมาะสำหรับการวิเคราะห์เชิงสถิติ

อย่างไรก็ตาม คอลัมน์เหล่านี้ยังคงถูกเก็บไว้ใน dataset เพื่อให้สามารถพิจารณาความสำคัญของตัวแปรในขั้นตอน EDA ต่อไป

# ตรวจสอบ Duplicate

In [ ]:
# ตรวจสอบแถวที่ซ้ำกันทั้งหมดใน DataFrame และแสดงจำนวน
print("Duplicate rows:", df.duplicated().sum())

dataset นี้ ไม่มีข้อมูลซ้ำ

# reset index หลังลบแถว

หลังจากลบแถว header ซ้ำและแถวสรุปรวม ทำให้ index ของข้อมูลไม่ต่อเนื่อง จึงทำการ reset index เพื่อให้ลำดับแถวเรียงใหม่

In [ ]:
# รีเซ็ต Index ของ DataFrame ให้เรียงลำดับใหม่และลบ Index เก่าทิ้งไป (drop=True)
df = df.reset_index(drop=True)

In [ ]:
# แสดง 5 แถวแรกของ DataFrame อีกครั้งหลังจากรีเซ็ต Index
df.head()

# ตรวจสอบสถิติพื้นฐาน

In [ ]:
df.describe()

ใช้ดู distribution ของข้อมูล เช่น
จำนวนเหตุไฟไหม้
จำนวนบ้านที่เสียหาย
มูลค่าความเสียหาย
เพื่อใช้เป็นพื้นฐานในการทำ Exploratory Data Analysis (EDA)

# ตรวจสอบรายชื่อจังหวัด

In [ ]:
df['province'].unique()

จากข้อมูลบนเว็บไซต์ระบุว่าชุดข้อมูลครอบคลุม 72 จังหวัด แต่เมื่อทำการตรวจสอบข้อมูลจริงพบว่ามีข้อมูลจังหวัดที่สามารถใช้วิเคราะห์ได้จำนวน 65 จังหวัด ดังนั้นการวิเคราะห์ในงานนี้จะใช้ข้อมูลจังหวัดทั้ง 65 จังหวัดเป็นขอบเขตของการศึกษา

## Cleaning Log

เพื่อให้ข้อมูลพร้อมสำหรับการวิเคราะห์ ได้มีการทำความสะอาดข้อมูลตามลำดับดังนี้

1. ตรวจสอบโครงสร้างและขนาดของข้อมูลเบื้องต้น พบว่า dataset มี 72 แถว และ 35 คอลัมน์  
2. ลบแถวแรกของข้อมูล เนื่องจากเป็น header ที่ซ้ำกับชื่อคอลัมน์  
3. ลบแถวสรุปรวมและแถวข้อความท้ายไฟล์ โดยคงไว้เฉพาะข้อมูลจังหวัดที่ใช้วิเคราะห์จริงจำนวน 65 แถว  
4. แทนค่าข้อมูลที่ระบุด้วยเครื่องหมาย `-` ให้เป็น `NaN` เพื่อจัดการ missing values ให้เป็นมาตรฐานเดียวกัน  
5. ลบ comma ออกจากค่าตัวเลข เช่น `6,738,000` เพื่อให้สามารถแปลงข้อมูลเป็นตัวเลขได้  
6. แปลงคอลัมน์เชิงปริมาณให้อยู่ในรูปแบบ numeric สำหรับใช้ในการคำนวณและสร้างกราฟ  
7. ตรวจสอบค่าที่หายไป (Missing Values) ในแต่ละคอลัมน์ เพื่อประเมินความพร้อมของข้อมูลก่อนทำ EDA  
8. ตรวจสอบข้อมูลซ้ำ (Duplicate) และไม่พบข้อมูลซ้ำที่ต้องลบออก  
9. Reset index ใหม่หลังจากลบแถวที่ไม่เกี่ยวข้อง เพื่อให้ลำดับข้อมูลต่อเนื่องและพร้อมใช้งาน

### 2️⃣ Exploratory Data Analysis (EDA)

# 1. กำหนดตัวแปรสำคัญที่จะใช้วิเคราะห์

In [ ]:
print(df.columns.tolist())

ขั้นตอนนี้ใช้เพื่อตรวจสอบชื่อคอลัมน์ทั้งหมดในชุดข้อมูล และเลือกตัวแปรสำคัญที่เกี่ยวข้องกับเหตุอัคคีภัย ความเสียหาย และผลกระทบต่อชีวิตและทรัพย์สิน

# 2. สรุป KPI ของชุดข้อมูล

In [ ]:
# ตัวอย่าง KPI เบื้องต้น: คำนวณจำนวนเหตุการณ์ทั้งหมด, มูลค่าความเสียหาย, จำนวนผู้บาดเจ็บและเสียชีวิต
total_incidents = df['event_count'].sum()
total_damage = df['property_damage_cost'].sum()
total_injured = df['injuries_count'].sum()
total_deaths = df['fatalities_count'].sum()

print("จำนวนครั้งที่เกิดเหตุการณ์ทั้งหมด:", total_incidents)
print("มูลค่าความเสียหายด้านทรัพย์สินของราษฎร(บาท)ทั้งหมด:", total_damage)
print("ราษฎรบาดเจ็บ(คน)ทั้งหมด:", total_injured)
print("ราษฎรเสียชีวิต(คน)ทั้งหมด:", total_deaths)

KPI เป็นตัวเลขสำคัญที่ช่วยสรุปภาพรวมของชุดข้อมูล

เช่น

จำนวนเหตุอัคคีภัยทั้งหมด

มูลค่าความเสียหายรวม

จำนวนผู้บาดเจ็บรวม

จำนวนผู้เสียชีวิตรวม

# 3. ดูสถิติพื้นฐานของตัวแปรเชิงปริมาณ

In [ ]:
df.describe().T

ใช้ดูค่าพื้นฐานของข้อมูล เช่น ค่าเฉลี่ย ค่าต่ำสุด ค่าสูงสุด และการกระจายตัวเบื้องต้นของแต่ละตัวแปร เพื่อใช้เลือกตัวแปรที่น่าสนใจไปวิเคราะห์ต่อ

# 4. วิเคราะห์ Distribution ของตัวแปรสำคัญ

In [ ]:
numeric_cols = ['event_count', 'property_damage_cost', 'injuries_count', 'fatalities_count']

for col in numeric_cols:
    # สร้างกราฟฮิสโตแกรมเพื่อดูการกระจายตัวของข้อมูลในแต่ละคอลัมน์
    plt.figure(figsize=(8,5))
    plt.hist(df[col].dropna(), bins=20)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

จากการวิเคราะห์ distribution ของตัวแปรสำคัญทั้ง 4 ตัว ได้แก่ จำนวนเหตุอัคคีภัย มูลค่าความเสียหาย จำนวนผู้บาดเจ็บ และจำนวนผู้เสียชีวิต พบว่าข้อมูลส่วนใหญ่มีการกระจุกตัวอยู่ในค่าระดับต่ำและมีลักษณะเบ้ขวา แสดงให้เห็นว่าจังหวัดส่วนใหญ่มีเหตุการณ์และผลกระทบไม่สูงมาก แต่มีบางจังหวัดที่มีค่าสูงผิดปกติอย่างชัดเจน ซึ่งสะท้อนถึงความแตกต่างของระดับความรุนแรงของอัคคีภัยในแต่ละพื้นที่ และสามารถนำไปสู่การวิเคราะห์เชิงลึกเกี่ยวกับพื้นที่เสี่ยงต่อไป

# 5. ตรวจสอบ Outlier ด้วย Boxplot

เลือก boxplot เพราะอยากดูว่าในข้อมูลมี outlier หรือไม่ และจังหวัดไหนมีค่าที่โดดออกจากกลุ่มหลักอย่างชัดเจน

In [ ]:
for col in numeric_cols:
    # สร้าง Boxplot เพื่อตรวจสอบค่า Outlier ในแต่ละคอลัมน์
    plt.figure(figsize=(8,4))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

**Boxplot ของจำนวนเหตุอัคคีภัย**แสดงให้เห็นว่าข้อมูลส่วนใหญ่กระจุกอยู่ในช่วงค่าต่ำถึงปานกลาง และมีค่าผิดปกติด้านขวาหลายจุด โดยเฉพาะบางจังหวัดที่มีจำนวนเหตุสูงมากกว่ากลุ่มอื่นอย่างชัดเจน สะท้อนว่าความถี่ในการเกิดอัคคีภัยไม่ได้กระจายเท่ากันในทุกพื้นที่

**Boxplot ของมูลค่าความเสียหาย**แสดงให้เห็นว่าข้อมูลส่วนใหญ่กระจุกตัวอยู่ในช่วงค่าต่ำ ขณะที่มีค่าผิดปกติหลายจุดทางด้านขวา และมีบางจังหวัดที่มีมูลค่าความเสียหายสูงมากเป็นพิเศษ สะท้อนว่าความเสียหายจากอัคคีภัยไม่ได้เกิดขึ้นในระดับใกล้เคียงกันทุกพื้นที่ แต่มีบางจังหวัดที่ได้รับผลกระทบทางเศรษฐกิจรุนแรงกว่ากลุ่มอื่นอย่างชัดเจน

**Boxplot ของจำนวนผู้บาดเจ็บ**แสดงให้เห็นว่าข้อมูลส่วนใหญ่กระจุกอยู่ในค่าต่ำ แต่มีค่าผิดปกติหลายจุดทางด้านขวา โดยเฉพาะบางจังหวัดที่มีจำนวนผู้บาดเจ็บสูงกว่ากลุ่มหลักอย่างมาก สะท้อนถึงความแตกต่างด้านความรุนแรงของเหตุอัคคีภัยในแต่ละพื้นที่

**Boxplot ของจำนวนผู้เสียชีวิตแสดงใ**ห้เห็นว่าข้อมูลส่วนใหญ่กระจุกตัวอยู่ในระดับต่ำมาก และมีค่าผิดปกติเพียงบางจุดที่สูงกว่ากลุ่มหลัก แม้การกระจายตัวของตัวแปรนี้จะไม่สูงมาก แต่จังหวัดที่มีจำนวนผู้เสียชีวิตมากกว่าปกติยังถือเป็นพื้นที่ที่ควรเฝ้าระวังเป็นพิเศษ

# 6. จังหวัดที่เกิดเหตุไฟไหม้สูงสุด

In [ ]:
top_fire = df.sort_values('event_count', ascending=False).head(10)

# สร้างกราฟแท่งแสดง 10 จังหวัดที่มีเหตุการณ์ไฟไหม้สูงสุด
plt.figure(figsize=(10,6))
plt.bar(top_fire['province'], top_fire['event_count'])
plt.title('Top 10 Provinces with Highest Fire Incidents')
plt.xlabel('Province')
plt.ylabel('Fire Incidents')
plt.xticks(rotation=45)
plt.show()

top_fire[['province', 'event_count']]

จะเห็นว่า อันดับ 1 คือ จังหวัด นนทบุรี
อันดับ 2 คือ จังหวัดปทุมธานี
และ อันดับ 3 คือ บุรีรัมย์ ซึ่งทั้งสามจังหวัดนี้อาจจะมีเหตุผลหรืออะไรบางอย่างที่ทำให้มีเหตุการณ์อัคคีภัยสูงสุดติดสามอันดับในประเทศไทย

# 7. จังหวัดที่มีมูลค่าความเสียหายสูงสุด

In [ ]:
top_damage = df.sort_values('property_damage_cost', ascending=False).head(10)

# สร้างกราฟแท่งแสดง 10 จังหวัดที่มีมูลค่าความเสียหายสูงสุด
plt.figure(figsize=(10,6))
plt.bar(top_damage['province'], top_damage['property_damage_cost'])
plt.title('Top 10 Provinces with Highest Estimated Damage')
plt.xlabel('Province')
plt.ylabel('property_damage_cost')
plt.xticks(rotation=45)
plt.show()

top_damage[['province', 'property_damage_cost']]

แม้ว่าบางจังหวัดอาจเกิดเหตุไฟไหม้ไม่บ่อย แต่หากมีความเสียหายสูงมาก ก็ถือเป็นจังหวัดที่ควรให้ความสำคัญในการบริหารความเสี่ยงเช่นกัน

# 8. ความสัมพันธ์ระหว่างจำนวนเหตุไฟไหม้กับมูลค่าความเสียหาย

In [ ]:
# สร้าง scatter plot เพื่อดูความสัมพันธ์ระหว่างจำนวนเหตุไฟไหม้กับมูลค่าความเสียหาย
plt.figure(figsize=(8,6))
plt.scatter(df['event_count'], df['property_damage_cost'])
plt.title('Fire Incidents vs Estimated Damage')
plt.xlabel('Fire Incidents')
plt.ylabel('Estimated Damage')
plt.show()

จากกราฟความสัมพันธ์ระหว่างจำนวนเหตุการณ์ไฟไหม้และมูลค่าความเสียหายโดยประมาณ พบว่าข้อมูลส่วนใหญ่กระจุกตัวอยู่ในช่วงจำนวนเหตุการณ์ต่ำถึงปานกลางและมีความเสียหายไม่สูงมากนัก อย่างไรก็ตาม มีบางจุดที่มีมูลค่าความเสียหายสูงผิดปกติแม้จำนวนเหตุการณ์จะไม่ได้สูงที่สุด แสดงให้เห็นว่าจำนวนเหตุการณ์ไฟไหม้ไม่ได้เป็นตัวกำหนดมูลค่าความเสียหายโดยตรงเสมอไป และความเสียหายอาจขึ้นอยู่กับปัจจัยอื่นร่วมด้วย เช่น ความรุนแรงของเหตุการณ์ ลักษณะพื้นที่ และมูลค่าทรัพย์สินที่ได้รับผลกระทบ

# 9. วิเคราะห์ความสัมพันธ์เชิงสหสัมพันธ์ (Correlation)

In [ ]:
# คำนวณ Correlation Matrix สำหรับคอลัมน์ที่เป็นตัวเลขทั้งหมด
corr = df.select_dtypes(include='number').corr()
corr

In [ ]:
plt.figure(figsize=(24,16))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

 # 🔍 จุดที่น่าสนใจ (Key Insights)

**ความสูญเสียชีวิตและมูลค่าความเสียหาย:** ตัวแปร fatalities_count (จำนวนผู้เสียชีวิต) และ property_damage_cost (มูลค่าความเสียหาย) มีความสัมพันธ์เชิงบวกกันสูงมากถึง 0.84 สิ่งนี้บ่งชี้ว่าเหตุการณ์ที่มีผู้เสียชีวิตมักจะเป็นเหตุการณ์เพลิงไหม้ขนาดใหญ่ที่สร้างความเสียหายต่อทรัพย์สินอย่างรุนแรงตามไปด้วย

**จำนวนครั้งที่เกิดเหตุและความเสียหายของบ้านเรือน:** event_count มีความสัมพันธ์กับ partial_home_loss (บ้านเรือนเสียหายบางส่วน) สูงถึง 0.86 แสดงให้เห็นว่ายิ่งเกิดเหตุบ่อย ความเสียหายประเภทนี้ก็ยิ่งแปรผันตามอย่างชัดเจน

**กลุ่มโรงงานและโกดัง:** ตัวแปร factories, warehouses และ property_damage_cost มีความสัมพันธ์เชิงบวกต่อกันในระดับสูง (เช่น factories กับ property_damage_cost อยู่ที่ 0.77) สะท้อนให้เห็นว่าหากเกิดเหตุในพื้นที่อุตสาหกรรม มูลค่าความเสียหายมักจะพุ่งสูงกว่าพื้นที่ประเภทอื่น

# 10. สรุป Insight เบื้องต้นจาก EDA

### Preliminary Insights from EDA

จากการสำรวจข้อมูลเบื้องต้น พบว่า
1. จำนวนเหตุอัคคีภัยของแต่ละจังหวัดมีการกระจายตัวไม่เท่ากัน โดยมีบางจังหวัดที่เกิดเหตุสูงกว่าจังหวัดอื่นอย่างชัดเจน
2. มูลค่าความเสียหายมีลักษณะกระจายแบบเบ้ขวา สะท้อนว่าหลายจังหวัดมีความเสียหายไม่สูงมาก แต่บางจังหวัดมีความเสียหายสูงผิดปกติ
3. จำนวนเหตุไฟไหม้และมูลค่าความเสียหายมีความสัมพันธ์กันในระดับหนึ่ง แต่ไม่ใช่ทุกจังหวัดที่เกิดเหตุบ่อยแล้วจะเสียหายสูงที่สุด
4. ข้อมูลมีความเหมาะสมสำหรับการวิเคราะห์เชิงลึกต่อในประเด็นพื้นที่เสี่ยง ความรุนแรงของความเสียหาย และผลกระทบต่อชีวิตและทรัพย์สิน

## 3️⃣ Insight Discovery

🔥 ข้อ 1 — จังหวัดไหนเสี่ยงจริง? (True Risk Profile)
📊 จากข้อมูล จำนวนเหตุการณ์อัคคีภัย และ มูลค่าความเสียหายจากเหตุการณ์

Quadrant 1: High Event + High Damage
เสี่ยงสูงจริง
จังหวัดกลุ่มนี้มีทั้งจำนวนเหตุการณ์สูงและมูลค่าความเสียหายสูง
จึงควรได้รับการจัดลำดับความสำคัญสูงสุดในการป้องกัน เฝ้าระวัง และจัดสรรทรัพยากร

Quadrant 2: High Event + Low Damage
เกิดบ่อยแต่ความรุนแรงต่อครั้งต่ำกว่า
จังหวัดกลุ่มนี้ควรเน้นมาตรการป้องกันเชิงพื้นที่ การลดความถี่ และการควบคุมต้นเหตุ

Quadrant 3: Low Event + High Damage
เกิดไม่บ่อยแต่เสียหายหนัก
จังหวัดกลุ่มนี้น่าสนใจมาก เพราะแม้เหตุการณ์ไม่ถี่ แต่เมื่อเกิดขึ้นแล้วมีผลกระทบรุนแรง
อาจต้องเน้นมาตรการรับมือเหตุรุนแรง การเตรียมพร้อม และการป้องกันในพื้นที่ทรัพย์สินมูลค่าสูง

Quadrant 4: Low Event + Low Damage
ความเสี่ยงต่ำกว่ากลุ่มอื่น
ยังควรติดตาม แต่ไม่ใช่กลุ่มแรกในการจัดสรรทรัพยากร

In [ ]:
# กำหนดค่ามัธยฐานของจำนวนเหตุการณ์และมูลค่าความเสียหายเพื่อใช้เป็นเส้นแบ่ง Quadrant
x_median = df['event_count'].median()
y_median = df['property_damage_cost'].median()

plt.figure(figsize=(12,8))

# สร้าง scatter plot
plt.scatter(df['event_count'], df['property_damage_cost'], alpha=0.7)

# ใส่ชื่อจังหวัดสำหรับบางจุดที่โดดเด่น (หากมีการกำหนด 'highlight' มาก่อน)
# for i, row in df.iterrows():
#     plt.text(row['event_count'], row['property_damage_cost'], row['province_en'], fontsize=8)

# เส้นแบ่ง quadrant
plt.axvline(x=x_median, linestyle='--')
plt.axhline(y=y_median, linestyle='--')

plt.title('True Risk Profile of Fire Incidents by Province')
plt.xlabel('Number of Fire Incidents')
plt.ylabel('Property Damage Cost')
plt.show()

# คอลัมน์กลุ่มความเสี่ยง

In [ ]:
# สร้างฟังก์ชันสำหรับจำแนกกลุ่มความเสี่ยงของแต่ละจังหวัด
def classify_risk(row):
    if row['event_count'] >= x_median and row['property_damage_cost'] >= y_median:
        return 'High Event / High Damage'
    elif row['event_count'] >= x_median and row['property_damage_cost'] < y_median:
        return 'High Event / Low Damage'
    elif row['event_count'] < x_median and row['property_damage_cost'] >= y_median:
        return 'Low Event / High Damage'
    else:
        return 'Low Event / Low Damage'

# สร้างคอลัมน์ 'risk_group' โดยใช้ฟังก์ชัน classify_risk
df['risk_group'] = df.apply(classify_risk, axis=1)
# แสดงคอลัมน์ที่เกี่ยวข้องกับกลุ่มความเสี่ยง
df[['province', 'event_count', 'property_damage_cost', 'risk_group']]

# ดูจำนวนจังหวัดในแต่ละกลุ่ม:

In [ ]:
# นับจำนวนจังหวัดในแต่ละกลุ่มความเสี่ยง
df['risk_group'].value_counts()

# ใช้สีแยกกลุ่ม

In [ ]:
colors = {
    'High Event / High Damage': 'red',
    'High Event / Low Damage': 'orange',
    'Low Event / High Damage': 'purple',
    'Low Event / Low Damage': 'green'
}

plt.figure(figsize=(12,8))

for group, color in colors.items():
    subset = df[df['risk_group'] == group]
    plt.scatter(subset['event_count'], subset['property_damage_cost'],
                color=color, label=group, alpha=0.7)

# กำหนดจังหวัดที่จะไฮไลท์ (เช่น 10 จังหวัดที่มีความเสียหายสูงสุดหรือเหตุการณ์สูงสุด)
highlight = df.nlargest(10, 'property_damage_cost').index.union(df.nlargest(10, 'event_count').index)

for i, row in df.loc[highlight].iterrows():
    plt.text(row['event_count'], row['property_damage_cost'], row['province_en'], fontsize=9)

plt.axvline(x=x_median, linestyle='--')
plt.axhline(y=y_median, linestyle='--')

plt.title('True Risk Profile of Fire Incidents by Province')
plt.xlabel('Number of Fire Incents')
plt.ylabel('Property Damage Cost')
plt.legend()
plt.show()

# top 5 จังหวัดในกลุ่ม High Event / High Damage

In [ ]:
# กรองและเรียงลำดับจังหวัดในกลุ่ม 'High Event / High Damage' เพื่อดูจังหวัดที่สำคัญที่สุด
priority = df[df['risk_group'] == 'High Event / High Damage'][
    ['province', 'event_count', 'property_damage_cost']
].sort_values(by=['property_damage_cost', 'event_count'], ascending=False)

priority.head(5)

### Insight 1: จังหวัดไหนเสี่ยงจริง?

จากการเปรียบเทียบจำนวนเหตุการณ์อัคคีภัยกับมูลค่าความเสียหาย พบว่า จังหวัดที่มีความเสี่ยงสูงจริงคือจังหวัดที่อยู่ในกลุ่ม **High Event /HighDamage**ซึ่งมีทั้งความถี่ของเหตุการณ์สูงและผลกระทบทางเศรษฐกิจรุนแรงพร้อมกัน โดย

จังหวัดเด่นในกลุ่มนี้ ได้แก่ **ชลบุรี นนทบุรี นครปฐม อุบลราชธานี และนครราชสีมา**

จังหวัดเหล่านี้ควรได้รับการจัดลำดับความสำคัญก่อนในการจัดสรรทรัพยากรและมาตรการป้องกันอัคคีภัย

ในขณะเดียวกัน **ยังพบกลุ่มจังหวัดที่มีจำนวนเหตุสูงแต่ความเสียหายต่ำ เช่น ปทุมธานี สกลนคร และสมุทรปราการ**

ซึ่งสะท้อนว่าปัญหาหลักอาจอยู่ที่ความถี่ของการเกิดเหตุ

ขณะที**จังหวัดบางแห่ง เช่น กาญจนบุรี หนองบัวลำภู และบึงกาฬ แม้เกิดเหตุไม่บ่อยแต่กลับมีมูลค่าความเสียหายสูง**แสดงถึงความรุนแรงของเหตุการณ์ต่อครั้ง

อย่างไรก็ตาม การตีความผลลัพธ์นี้ควรพิจารณาร่วมกับข้อจำกัดของข้อมูล เนื่องจากหลายจังหวัดไม่มีข้อมูลมูลค่าความเสียหาย ทำให้การจัดกลุ่มความเสี่ยงในบางกรณีอาจต่ำกว่าความเป็นจริง

เลือกใช้ Scatter Plot ร่วมกับ Quadrant Analysis ในข้อ 1 เพราะโจทย์ต้องการให้ดู 2 มิติพร้อมกัน คือ จำนวนเหตุการณ์และมูลค่าความเสียหาย กราฟนี้เหมาะที่สุดสำหรับการมองเห็นความสัมพันธ์ระหว่างสองตัวแปร และเมื่อเพิ่มเส้นแบ่ง quadrant ก็ช่วยแยกจังหวัดออกเป็น 4 กลุ่มความเสี่ยงได้อย่างชัดเจน เช่น เหตุสูง-เสียหายสูง หรือ เหตุต่ำ-เสียหายสูง

#📍🏚️ ข้อ 2 — ความเสียหายกระจุกตัวที่ไหน และหนักแค่ไหน? (Where & How Severe?)


ทำ Heatmap แสดงพื้นที่แต่ละที่่มีความเสียหายมากน้อยเพียงใด

1) Where?
ความเสียหายกระจุกตัวอยู่ที่ ภูมิภาคไหน
2) What type?
สิ่งปลูกสร้างประเภทไหนเสียหายมากที่สุด
3) How severe?
มูลค่าความเสียหายสูงแค่ไหน เมื่อเทียบกับภูมิภาคอื่น
4) Hotspot
ช่องไหนใน heatmap เข้มสุด = จุดวิกฤต

# Step 1: สร้างคอลัมน์ภูมิภาคจากจังหวัด

In [ ]:
region_map = {
    'เชียงใหม่': 'North', 'เชียงราย': 'North', 'ลำปาง': 'North', 'ลำพูน': 'North', 'แพร่': 'North',
    'น่าน': 'North', 'พะเยา': 'North', 'แม่ฮ่องสอน': 'North', 'อุตรดิตถ์': 'North',

    'ขอนแก่น': 'Northeast', 'กาฬสินธุ์': 'Northeast', 'นครราชสีมา': 'Northeast',
    'อุบลราชธานี': 'Northeast', 'อุดรธานี': 'Northeast', 'บุรีรัมย์': 'Northeast',
    'สุรินทร์': 'Northeast', 'ศรีสะเกษ': 'Northeast', 'สกลนคร': 'Northeast',
    'มหาสารคาม': 'Northeast', 'มุกดาหาร': 'Northeast', 'ยโสธร': 'Northeast',
    'ร้อยเอ็ด': 'Northeast', 'หนองบัวลำภู': 'Northeast', 'บึงกาฬ': 'Northeast',
    'ชัยภูมิ': 'Northeast', 'เลย': 'Northeast', 'นครพนม': 'Northeast',
    'อำนาจเจริญ': 'Northeast',

    'นนทบุรี': 'Central', 'นครปฐม': 'Central', 'ปทุมธานี': 'Central', 'พระนครศรีอยุธยา': 'Central',
    'สระบุรี': 'Central', 'สุพรรณบุรี': 'Central', 'สิงห์บุรี': 'Central', 'อ่างทอง': 'Central',
    'ชัยนาท': 'Central', 'ลพบุรี': 'Central', 'นครสวรรค์': 'Central', 'อุทัยธานี': 'Central',
    'พิจิตร': 'Central', 'พิษณุโลก': 'Central', 'สุโขทัย': 'Central', 'เพชรบูรณ์': 'Central',
    'สมุทรปราการ': 'Central', 'สมุทรสาคร': 'Central', 'สมุทรสงคราม': 'Central',

    'ชลบุรี': 'East', 'ระยอง': 'East', 'จันทบุรี': 'East', 'ตราด': 'East',
    'ฉะเชิงเทรา': 'East', 'สระแก้ว': 'East', 'ปราจีนบุรี': 'East',

    'กาญจนบุรี': 'West', 'ราชบุรี': 'West', 'เพชรบุรี': 'West', 'ประจวบคีรีขันธ์': 'West',
    'ตาก': 'West',

    'กระบี่': 'South', 'ตรัง': 'South', 'นครศรีธรรมราช': 'South', 'นราธิวาส': 'South', 'ปัตตานี': 'South',
    'พังงา': 'South', 'พัทลุง': 'South', 'ยะลา': 'South', 'สงขลา': 'South', 'สตูล': 'South'
}

df['region'] = df['province'].map(region_map)
# สร้างคอลัมน์ 'region' ใน DataFrame โดยใช้ mapping จาก 'province' เพื่อระบุภูมิภาคของแต่ละจังหวัด

# Step 2: เลือกคอลัมน์ประเภทสิ่งปลูกสร้าง

In [ ]:
building_cols = {
    'total_home_loss': 'Full House Loss',
    'partial_home_loss': 'Partial House Loss',
    'high_rise_buildings': 'High-Rise',
    'farm_buildings': 'Farm/Nursery',
    'agricultural_land' : 'Agricultural',
    'department_stores': 'Dept. Stores',
    'rice_mills': 'Rice Mills',
    'shops': 'Shops',
    'temporary_shelter': 'Temporary Shelter',
    'landfill_area': 'Landfill',
    'religious_buildings': 'Religious',
    'schools': 'Schools',
    'commercial_buildings': 'Commercial',
    'factories': 'Factories',
    'government_offices': 'Gov. Offices',
    'warehouses': 'Warehouses'
}
# กำหนด dictionary เพื่อแปลงชื่อคอลัมน์ที่เกี่ยวข้องกับประเภทสิ่งปลูกสร้างให้เป็นชื่อที่อ่านง่ายขึ้น สำหรับใช้ในการวิเคราะห์ Heatmap และ Stacked Bar Chart

# Step 3: แปลง wide → long

In [ ]:
# ใช้ melt เพื่อแปลง DataFrame จาก wide format เป็น long format
long_df = df.melt(
    id_vars=['province', 'region'],
    value_vars=list(building_cols.keys()),
    var_name='building_type',
    value_name='damage_cost'
)

# แปลงชื่อ building_type ให้เป็นชื่อที่อ่านง่ายขึ้น
long_df['building_type'] = long_df['building_type'].map(building_cols)

# Step 4: ทำ Heatmap

In [ ]:
heatmap_data = long_df.pivot_table(index='region', columns='building_type', values='damage_cost', aggfunc='sum').fillna(0)

In [ ]:
plt.figure(figsize=(22, 12))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='OrRd', linewidths=0.5)
plt.title('Total Fire Damage Cost by Region and Building Type (Baht)')
plt.tight_layout()
plt.show()

จากกราฟพบว่า **ภาคตะวันออกเฉียงเหนือ (Northeast)**
มีความเสียหายกระจุกตัวเด่นที่สุดในหลายประเภทสิ่งปลูกสร้าง โดยเฉพาะ Full House Loss และ Partial House Loss ซึ่งมีค่าสูงสุดในกราฟ สะท้อนว่าอัคคีภัยในภูมิภาคนี้ส่งผลกระทบต่อ ภาคครัวเรือน อย่างรุนแรงมากกว่าภูมิภาคอื่น

**ขณะเดียวกัน ภาคกลาง (Central)**
ก็มีความเสียหายเด่นในหลายหมวดเช่นกัน โดยเฉพาะ Full House Loss, Partial House Loss, Commercial และ Shops แสดงให้เห็นว่าความเสียหายในภาคกลางไม่ได้กระทบเฉพาะบ้านเรือน แต่ยังเชื่อมโยงกับกิจกรรมเศรษฐกิจและพื้นที่ชุมชนด้วย

**ในส่วนของ ภาคใต้ (South)**
ความเสียหายเด่นในหมวด Agricultural และ Landfill มากกว่าหมวดอื่น สะท้อนว่ารูปแบบความเสียหายในภาคใต้มีลักษณะเฉพาะที่แตกต่างจากภาคกลางและภาคตะวันออกเฉียงเหนือ
โดยสรุป กราฟนี้ชี้ให้เห็นว่า รูปแบบความเสียหายจากอัคคีภัยไม่ได้เหมือนกันทุกภูมิภาค
* บางภูมิภาคเด่นที่บ้านเรือน
* บางภูมิภาคเด่นที่โครงสร้างเชิงเกษตรหรือกิจกรรมชุมชน

ดังนั้นการออกแบบมาตรการป้องกันไม่ควรใช้แนวทางเดียวทั้งประเทศ แต่ควรปรับตามประเภทสิ่งปลูกสร้างที่เป็น hotspot ของแต่ละภูมิภาค

เลือกใช้ Heatmap ในข้อ 2 เพราะต้องการแสดงข้อมูล 3 มิติพร้อมกัน ได้แก่ ภูมิภาค ประเภทสิ่งปลูกสร้าง และระดับความเสียหายหรือจำนวนสิ่งปลูกสร้างที่เสียหาย กราฟนี้ช่วยให้เห็น “จุดเข้ม” หรือ hotspot ได้รวดเร็ว และเหมาะมากเมื่อเราต้องเปรียบเทียบหลายหมวดหมู่ข้ามหลายพื้นที่พร้อมกัน

In [ ]:
def plot_pie_by_region(selected_region):
    region_df = long_df[long_df['region'] == selected_region]

    pie_data = (
        region_df.groupby('building_type')['damage_cost']
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )

    plt.figure(figsize=(8,8))
    plt.pie(
        pie_data,
        labels=pie_data.index,
        autopct='%1.1f%%',
        startangle=90
    )
    plt.title(f'Top 5 Damaged Structure Types in {selected_region}')
    plt.show()

interact(
    plot_pie_by_region,
    selected_region=sorted(long_df['region'].dropna().unique())
)

ยกตัวอย่าง ภาคกลางจากกราฟวงกลมนี้

สัดส่วนของ**สิ่งปลูกสร้างที่ได้รับความเสียหายมากที่สุด 5 อันดับแรกในภาคกลาง**
เพื่อให้เห็นว่าอัคคีภัยในภูมิภาคนี้กระทบกับอาคารประเภทใดมากที่สุดเมื่อเทียบกันภายในภูมิภาคเดียว

จากกราฟพบว่า ความเสียหายในภาคกลางกระจุกตัวมากที่สุดใน **Full House Loss และ Partial House Loss**ซึ่งมีสัดส่วนเท่ากันที่ 28.8% ต่อประเภท แสดงให้เห็นว่าความเสียหายจากอัคคีภัยในภาคกลางส่งผลกระทบต่อ ภาคครัวเรือน เป็นหลัก โดยมีทั้งกรณีที่บ้านเสียหายทั้งหลังและเสียหายบางส่วนในระดับใกล้เคียงกัน

**รองลงมาคือ Factories ที่ 15.4%**
สะท้อนว่าอัคคีภัยในภาคกลางไม่ได้กระทบเฉพาะบ้านเรือน แต่ยังมีผลต่อโครงสร้างที่เกี่ยวข้องกับกิจกรรมการผลิตด้วย

**ส่วน Commercial และ Shops มีสัดส่วนเท่ากันที่ 13.5%**
แสดงว่าความเสียหายยังเชื่อมโยงกับพื้นที่เศรษฐกิจและกิจกรรมการค้าภายในชุมชน

โดยสรุป กราฟนี้ชี้ให้เห็นว่า รูปแบบความเสียหายในภาคกลางมีทั้งมิติของครัวเรือนและเศรษฐกิจชุมชน แต่กลุ่มที่ได้รับผลกระทบมากที่สุดยังคงเป็น บ้านเรือน ดังนั้น หากต้องออกแบบมาตรการป้องกันในภาคกลาง ควรให้ความสำคัญกับการลดความเสี่ยงในที่อยู่อาศัยเป็นอันดับแรก ควบคู่กับการเฝ้าระวังในโรงงานและพื้นที่การค้า

เลือกใช้ pie chart เพราะต้องการแสดง สัดส่วนภายในภูมิภาคเดียว ว่าอาคารประเภทใดมีสัดส่วนความเสียหายมากที่สุด ทำให้มองเห็นภาพรวมขององค์ประกอบความเสียหายในภาคกลางได้รวดเร็วและเข้าใจง่าย

In [ ]:
treemap_data = (
    long_df.groupby(['region', 'building_type'])['damage_cost']
    .sum()
    .reset_index(name='count_damaged')
    .sort_values('count_damaged', ascending=False)
    .head(12)
)

treemap_data['label'] = (
    treemap_data['region'] + '\n' +
    treemap_data['building_type'] + '\n' +
    treemap_data['count_damaged'].astype(str)
)

plt.figure(figsize=(16, 9))
squarify.plot(
    sizes=treemap_data['count_damaged'],
    label=treemap_data['label'],
    alpha=0.8,
    text_kwargs={'fontsize':11}
)

plt.title('Top Fire Damage Hotspots by Region and Building Type')
plt.axis('off')
plt.show()

กราฟนี้แสดง hotspot หลักของความเสียหายจากอัคคีภัย โดยดูพร้อมกันทั้งภูมิภาคและประเภทสิ่งปลูกสร้าง จะเห็นชัดว่าความเสียหายกระจุกตัวมากที่สุดในกลุ่ม Agricultural ของภาคตะวันออกเฉียงเหนือและภาคใต้ รองลงมาคือความเสียหายของบ้านเรือนในภาคตะวันออกเฉียงเหนือและภาคกลาง

เลือกใช้ treemap เพราะต้องการแสดงให้เห็นพร้อมกันว่า “ความเสียหายกระจุกที่ไหน” และ “เกิดกับสิ่งปลูกสร้างประเภทใดมากที่สุด” โดยขนาดของบล็อกช่วยทำให้เห็น hotspot หลักได้ชัดเจนในภาพเดียว และเปรียบเทียบ combination ระหว่างภูมิภาคกับประเภทอาคารได้ง่ายกว่ากราฟทั่วไป

In [ ]:
# Normalized stacked bar — เห็น "สัดส่วน" ของแต่ละภูมิภาค ไม่ใช่แค่ตัวเลขรวม
# Calculate total damage for each building type
total_damage_by_building_type = long_df.groupby('building_type')['damage_cost'].sum().sort_values(ascending=False)

top5 = total_damage_by_building_type.head(5).index.tolist()

# Filter long_df to include only the top 5 building types by damage cost
filtered_long_df = long_df[long_df['building_type'].isin(top5)]

norm_data = filtered_long_df.pivot_table(
    index='region',
    columns='building_type',
    values='damage_cost',
    aggfunc='sum'
).fillna(0)

# แปลงเป็น % ต่อแถว
norm_pct = norm_data.div(norm_data.sum(axis=1), axis=0) * 100

norm_pct.plot(kind='bar', stacked=True, figsize=(14, 8), colormap='tab10')
plt.title('Damage Pattern by Region (% of Total Damage per Region)')
plt.xlabel('Region')
plt.ylabel('Proportion (%)')
plt.xticks(rotation=45)
plt.legend(title='Building Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

**ภาคกลาง — เสียหายกระจาย หลายประเภท**
Full House Loss ~25%, Partial House Loss ~25%, Agricultural ~41% สะท้อนว่าภาคกลางมีทั้งความเสียหายที่อยู่อาศัยและเกษตรในระดับใกล้เคียงกัน ไม่มีประเภทเดียวที่ครองสัดส่วนสูงโดดออกมา

**ภาคตะวันออก — บ้านเรือนเป็นหลัก**

Partial House Loss ครองสัดส่วนสูงสุด ~54% รวมกับ Full House Loss รวมกันเกือบ 80% แสดงว่าอัคคีภัยในภาคตะวันออกกระทบ ที่อยู่อาศัยเป็นหลัก เกษตรน้อยมาก (8%)
North — บ้านเรือนเป็นหลัก คล้าย East
Partial House Loss ~33%, Full House Loss ~41% รวมกันราว 74% รูปแบบคล้ายภาคตะวันออก แต่มี Agricultural สูงกว่าเล็กน้อย (17%)

**ภาคตะวันออกเฉียงเหนือ — Agricultural ครองชัดเจน**
Agricultural สูงถึง ~70% นี่คือจุดที่ต่างจากทุกภูมิภาคอย่างเห็นได้ชัด สะท้อนว่าไฟไหม้ในภาคอีสานกระทบ พื้นที่เกษตรกรรมเป็นหลัก มากกว่าที่อยู่อาศัย

**ภาคใต้ — Agricultural ครองชัดเจนเช่นกัน**
Agricultural ~94% สูงที่สุดในทุกภูมิภาค แทบไม่มีประเภทอื่นเลย บ่งชี้ว่าไฟในภาคใต้เผาพื้นที่เกษตรเกือบทั้งหมด

**ภาคตะวันตก — Full House Loss เป็นหลัก**
Full House Loss ~78% สูงมากเมื่อเทียบกับภูมิภาคอื่น แสดงว่าไฟในภาคตะวันตกเมื่อเกิดขึ้นมักเผาบ้านทั้งหลัง ไม่ใช่เสียหายบางส่วน

เหตุที่เลือกกราฟStacked Bar เพราะ Stacked Bar แสดงสัดส่วนให้เห็น pattern ทุกภูมิภาคเคียงกัน ต่างกันชัดเจน

In [ ]:
import geopandas as gpd

# โหลด shapefile หรือ geojson จังหวัดไทย
gdf = gpd.read_file('/content/thailand-provinces.topojson')

# ตรวจสอบคอลัมน์ที่มีอยู่ใน GeoDataFrame (gdf)
print(gdf.columns)
gdf.head()

In [ ]:
province_map_topo = {
    'กระบี่': 'Krabi',
    'กาญจนบุรี': 'Kanchanaburi',
    'กาฬสินธุ์': 'Kalasin',
    'กำแพงเพชร': 'Kamphaeng Phet',
    'ขอนแก่น': 'Khon Kaen',
    'จันทบุรี': 'Chanthaburi',
    'ฉะเชิงเทรา': 'Chachoengsao',
    'ชลบุรี': 'Chon Buri',
    'ชัยนาท': 'Chai Nat',
    'ชัยภูมิ': 'Chaiyaphum',
    'ชุมพร': 'Chumphon',
    'เชียงราย': 'Chiang Rai',
    'เชียงใหม่': 'Chiang Mai',
    'ตรัง': 'Trang',
    'ตราด': 'Trat',
    'ตาก': 'Tak',
    'นครนายก': 'Nakhon Nayok',
    'นครปฐม': 'Nakhon Pathom',
    'นครพนม': 'Nakhon Phanom',
    'นครราชสีมา': 'Nakhon Ratchasima',
    'นครศรีธรรมราช': 'Nakhon Si Thammarat',
    'นครสวรรค์': 'Nakhon Sawan',
    'นนทบุรี': 'Nonthaburi',
    'นราธิวาส': 'Narathiwat',
    'น่าน': 'Nan',
    'บึงกาฬ': 'Bueng Kan',   # ถ้า topojson ไม่มีจังหวัดนี้ อาจ merge ไม่ติด
    'บุรีรัมย์': 'Buri Ram',
    'ปทุมธานี': 'Pathum Thani',
    'ประจวบคีรีขันธ์': 'Prachuap Khiri Khan',
    'ปราจีนบุรี': 'Prachin Buri',
    'ปัตตานี': 'Pattani',
    'พระนครศรีอยุธยา': 'Phra Nakhon Si Ayutthaya',
    'พะเยา': 'Phayao',
    'พังงา': 'Phangnga',
    'พัทลุง': 'Phatthalung',
    'พิจิตร': 'Phichit',
    'พิษณุโลก': 'Phitsanulok',
    'เพชรบุรี': 'Phetchaburi',
    'เพชรบูรณ์': 'Phetchabun',
    'แพร่': 'Phrae',
    'ภูเก็ต': 'Phuket',
    'มหาสารคาม': 'Maha Sarakham',
    'มุกดาหาร': 'Mukdahan',
    'แม่ฮ่องสอน': 'Mae Hong Son',
    'ยะลา': 'Yala',
    'ยโสธร': 'Yasothon',
    'ระนอง': 'Ranong',
    'ระยอง': 'Rayong',
    'ราชบุรี': 'Ratchaburi',
    'ร้อยเอ็ด': 'Roi Et',
    'ลพบุรี': 'Lop Buri',
    'ลำปาง': 'Lampang',
    'ลำพูน': 'Lamphun',
    'เลย': 'Loei',
    'ศรีสะเกษ': 'Si Sa Ket',
    'สกลนคร': 'Sakon Nakhon',
    'สงขลา': 'Songkhla',
    'สตูล': 'Satun',
    'สมุทรปราการ': 'Samut Prakan',
    'สมุทรสงคราม': 'Samut Songkhram',
    'สมุทรสาคร': 'Samut Sakhon',
    'สระบุรี': 'Saraburi',
    'สระแก้ว': 'Sa Kaeo',
    'สิงห์บุรี': 'Sing Buri',
    'สุโขทัย': 'Sukhothai',
    'สุพรรณบุรี': 'Suphan Buri',
    'สุราษฎร์ธานี': 'Surat Thani',
    'สุรินทร์': 'Surin',
    'หนองคาย': 'Nong Khai',
    'หนองบัวลำภู': 'Nong Bua Lam Phu',
    'อ่างทอง': 'Ang Thong',
    'อำนาจเจริญ': 'Amnat Charoen',
    'อุดรธานี': 'Udon Thani',
    'อุตรดิตถ์': 'Uttaradit',
    'อุทัยธานี': 'Uthai Tani',
    'อุบลราชธานี': 'Ubon Ratchathani',
    'กรุงเทพมหานคร': 'Bangkok Metropolis'
}

df['province_en'] = df['province'].astype(str).str.strip().map(province_map_topo)
# สร้างคอลัมน์ 'province_en' ใน DataFrame โดยใช้การ map ชื่อจังหวัดภาษาไทยไปเป็นภาษาอังกฤษตาม province_map_topo

In [ ]:
import geopandas as gpd

# โหลด shapefile หรือ geojson จังหวัดไทย
gdf = gpd.read_file('/content/thailand-provinces.topojson')

# merge กับข้อมูล damage โดยใช้ชื่อจังหวัดภาษาอังกฤษ (province_en และ NAME_1)
map_df = gdf.merge(df, left_on='NAME_1', right_on='province_en', how='left')

# วาดแผนที่แสดงมูลค่าความเสียหายทางทรัพย์สินของราษฎร
fig, ax = plt.subplots(1, 1, figsize=(10, 14))
map_df.plot(
    column='property_damage_cost',
    cmap='OrRd',
    linewidth=0.5,
    edgecolor='black',
    legend=True,
    ax=ax,
    missing_kwds={'color': 'lightgrey', 'label': 'Missing data'}
)

ax.set_title('Thailand Choropleth Map of Fire Property Damage')
ax.axis('off')
plt.show()

 Thailand Choropleth Map
สิ่งที่เห็นจากแผนที่
ความเสียหายจากอัคคีภัย**วัดจาก มูลค่าความเสียหายด้านทรัพย์สินของราษฎร(บาท)**
ไม่ได้กระจายเท่ากันทั่วประเทศ
มีบางจังหวัดที่สีเข้มชัดมาก แสดงว่ามี มูลค่าความเสียหายสูงผิดปกติ
จังหวัดที่เด่นที่สุดคือ ชลบุรี ซึ่งเป็น hotspot หลักของประเทศ
นอกจากนี้ยังมีบางจังหวัดในภาคกลางและภาคตะวันออกเฉียงเหนือที่มีสีเข้มรองลงมา สะท้อนว่าความเสียหายไม่ได้กระจุกแค่พื้นที่เดียว แต่มีลักษณะกระจายเป็นบางจุด

ความเสียหายจากอัคคีภัยมีลักษณะ กระจุกตัวแบบ hotspot
จังหวัดที่เป็น hotspot ควรถูกจัดเป็นพื้นที่เป้าหมายลำดับแรกในการวางมาตรการป้องกัน
ในเชิงนโยบาย แผนที่นี้ช่วยตอบคำถามว่า “ควรเฝ้าระวังตรงไหนก่อน”

เลือกใช้ Choropleth Map เพราะโจทย์ข้อ 2 ต้องการตอบว่า ความเสียหายกระจุกตัวอยู่ที่ไหน ซึ่งเป็นคำถามเชิงพื้นที่โดยตรง แผนที่ทำให้เห็น hotspot ทางภูมิศาสตร์ได้ดีกว่ากราฟแท่งหรือกราฟเส้น เพราะผู้อ่านสามารถมองเห็นตำแหน่งของจังหวัดที่มีความเสียหายสูงได้ทันที

In [ ]:
# วาดแผนที่แสดงจำนวนผู้เสียชีวิตจากเหตุไฟไหม้
fig, ax = plt.subplots(1, 1, figsize=(10, 14))
map_df.plot(
    column='fatalities_count',
    cmap='OrRd',
    linewidth=0.5,
    edgecolor='black',
    legend=True,
    ax=ax,
    missing_kwds={'color': 'lightgrey', 'label': 'Missing data'}
)

ax.set_title('Thailand Choropleth Map of Fire Fatalities')
ax.axis('off')
plt.show()

กราฟแผนที่นี้แสดงให้เห็นว่า จำนวนผู้เสียชีวิตจากเหตุอัคคีภัยไม่ได้กระจายตัวเท่ากันทั่วประเทศ แต่กระจุกอยู่ในบางจังหวัดอย่างชัดเจน โดยจังหวัดที่มีสีเข้มที่สุดคือจังหวัดที่มีจำนวนผู้เสียชีวิตสูงสุดในชุดข้อมูล ขณะที่หลายจังหวัดเป็นสีเทา ซึ่งสะท้อนว่าไม่มีข้อมูลหรือไม่มีค่าที่ใช้แสดงผลในตัวแปรนี้
จากแผนที่จะเห็นว่า จุดร้อน (hotspots) ของการสูญเสียชีวิตไม่ได้อยู่เฉพาะในภูมิภาคเดียว แต่ปรากฏทั้งใน
* ภาคเหนือ
* ภาคกลาง
* และบางส่วนของ ภาคตะวันออกเฉียงเหนือ

ลักษณะนี้สะท้อนว่า “ความรุนแรงต่อชีวิต” ของอัคคีภัยอาจไม่ได้สัมพันธ์กับจำนวนเหตุเพียงอย่างเดียว แต่ขึ้นอยู่กับบริบทของพื้นที่และความเปราะบางของเหตุการณ์ในแต่ละจังหวัดด้วย

* ถ้าบางจังหวัดเป็นสีเทาเพราะไม่มีข้อมูล
ต้องระวังไม่ตีความว่า “ไม่มีความเสี่ยง”
เพราะมันคือ ข้อจำกัดของข้อมูล

* เลือกใช้ Choropleth Map เพราะโจทย์ข้อ 2 ต้องการตอบว่า ความเสียหายกระจุกตัวอยู่ที่ไหน ซึ่งเป็นคำถามเชิงพื้นที่โดยตรง

แผนที่ทำให้เห็น hotspot ทางภูมิศาสตร์ได้ดีกว่ากราฟแท่งหรือกราฟเส้น เพราะผู้อ่านสามารถมองเห็นตำแหน่งของจังหวัดที่มีความเสียหายสูงได้ทันที

# ⚠️📈 ข้อ 3 — อะไรคือตัวชี้วัดความรุนแรง? (What predicts severity?)

# Step 1: นิยาม severity

In [ ]:
# กำหนดค่า Threshold สำหรับมูลค่าความเสียหาย (ที่เปอร์เซ็นไทล์ที่ 75)
damage_threshold = df['property_damage_cost'].quantile(0.75)

# สร้างคอลัมน์ 'severe_event' เพื่อระบุว่าเหตุการณ์ใดเป็นเหตุการณ์รุนแรง
df['severe_event'] = (
    (df['fatalities_count'] >= 2) |  # มีผู้เสียชีวิตตั้งแต่ 2 คนขึ้นไป
    (df['injuries_count'] >= 10) |   # มีผู้บาดเจ็บตั้งแต่ 10 คนขึ้นไป
    (df['property_damage_cost'] >= damage_threshold) # มูลค่าความเสียหายสูงกว่าเปอร์เซ็นไทล์ที่ 75
).astype(int) # แปลงเป็น 0 หรือ 1

# แสดงตัวอย่างข้อมูล 5 แถวแรกของคอลัมน์ที่เกี่ยวข้อง
df[['province', 'fatalities_count', 'injuries_count', 'property_damage_cost', 'severe_event']].head()

กำหนดตัวชี้วัดความรุนแรงภายใต้เงือนไข ว่า ต้องมีจำนวนผู้เสียชีวิตมากกว่าหรือเท่ากับ 2 , จำนวนผู้บาดเจ็บต้องมากกว่าหรือเท่ากับ 10 หรือ

# Step 2: เลือกตัวแปรที่อยากทดสอบ
เลือกเฉพาะ numeric columns ที่มีเหตุผลเชิงนโยบาย

In [ ]:
# เลือกตัวแปรเชิงปริมาณที่ต้องการทดสอบความสัมพันธ์กับความรุนแรงของเหตุการณ์
candidate_vars = [
    'event_count',
    'injuries_count',
    'fatalities_count',
    'total_home_loss',
    'partial_home_loss',
    'shops',
    'factories',
    'commercial_buildings',
    'government_offices',
    'warehouses',
    'high_rise_buildings',
    'schools',
    'religious_buildings'
]

# Step 3: Correlation Heatmap
จะใช้ property_damage_cost เป็น severity

In [ ]:
# รวมตัวแปรที่เลือกและคอลัมน์ 'property_damage_cost' เพื่อสร้าง Correlation Matrix
analysis_cols = candidate_vars + ['property_damage_cost']
corr = df[analysis_cols].corr()

plt.figure(figsize=(24, 16))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Fire Severity Factors')
plt.tight_layout()
plt.show()

เลือกใช้ Correlation Heatmap ในข้อ 3 เพราะต้องการดูภาพรวมว่าตัวแปรใดมีความสัมพันธ์กับความรุนแรงของเหตุอัคคีภัยบ้าง กราฟนี้เหมาะสำหรับการเปรียบเทียบความสัมพันธ์ของหลายตัวแปรพร้อมกัน และช่วยคัดกรองว่าตัวแปรไหนควรนำไปพิจารณาต่อในเชิงนโยบาย

# Step 4 :สร้างกราฟแท่งแสดงตัวแปรที่มีความสัมพันธ์สูงสุดกับมูลค่าความเสียหาย


In [ ]:
# กรอง Correlation Matrix เพื่อหาตัวแปรที่มีความสัมพันธ์สูงสุดกับ 'property_damage_cost'
severity_corr = corr['property_damage_cost'].drop('property_damage_cost').sort_values(ascending=False)

# สร้างกราฟแท่งแสดงตัวแปรที่มีความสัมพันธ์สูงสุดกับมูลค่าความเสียหาย
plt.figure(figsize=(10, 6))
plt.bar(severity_corr.index, severity_corr.values)
plt.xticks(rotation=75, ha='right')
plt.ylabel('Correlation with Property Damage Cost')
plt.title('Top Factors Associated with Fire Severity')
plt.tight_layout()
plt.show()

severity_corr

* Fatalities
* Warehouses
* Factories

คือ 3 ปัจจัยที่สัมพันธ์กับ severity สูงสุด

**1) ตัวชี้วัดความรุนแรงที่เด่นที่สุดคือ “การสูญเสียชีวิต”**
fatalities_count มี correlation สูงสุดกับ property_damage_cost ที่ 0.841

แปลว่าเหตุการณ์ที่มีผู้เสียชีวิต มักสัมพันธ์กับเหตุที่มีความเสียหายทางทรัพย์สินสูงด้วย
แปลความ
เหตุที่รุนแรงต่อชีวิต มักรุนแรงต่อทรัพย์สินไปพร้อมกัน
ถ้าเหตุหนึ่งเริ่มมีสัญญาณเสี่ยงต่อการเสียชีวิต ก็มีแนวโน้มเป็นเหตุการณ์ใหญ่หรือ “หายนะ”

So what
ระบบเตือนภัยและระบบตอบสนองฉุกเฉินควรให้ความสำคัญสูงกับเหตุที่มีแนวโน้มกระทบชีวิต ไม่ใช่มองแค่จำนวนเหตุอย่างเดียว

* เลือกใช้ Ranked Bar Chart เพราะถึงแม้ heatmap จะเห็นภาพรวมได้ดี แต่ผู้กำหนดนโยบายต้องการคำตอบที่เร็วและชัดว่า ปัจจัยไหนสำคัญที่สุด การจัดเรียงค่าจากมากไปน้อยทำให้เห็นลำดับความสำคัญของตัวแปรได้ทันที และเหมาะกับการสื่อสารเชิงนโยบายมากกว่า heatmap เพียงอย่างเดียว

# “เหตุรุนแรงต่างจากเหตุไม่รุนแรงยังไง”

In [ ]:
# จัดกลุ่มข้อมูลตาม 'severe_event' และคำนวณค่าเฉลี่ยของตัวแปรที่เลือก
group_means = df.groupby('severe_event')[candidate_vars].mean().T
group_means.columns = ['Non-Severe', 'Severe']

# สร้างกราฟแท่งเปรียบเทียบค่าเฉลี่ยของตัวแปรในเหตุการณ์ที่ไม่รุนแรงและรุนแรง
group_means.plot(kind='bar', figsize=(12, 7))
plt.title('Average Characteristics of Severe vs Non-Severe Fire Events')
plt.ylabel('Average Value')
plt.xticks(rotation=75, ha='right')
plt.tight_layout()
plt.show()

เลือกใช้กราฟเปรียบเทียบระหว่าง เหตุรุนแรงกับเหตุไม่รุนแรง เพราะต้องการทำให้เห็นว่าลักษณะของทั้งสองกลุ่มต่างกันอย่างไร กราฟประเภทนี้ช่วยให้ผู้อ่านเข้าใจได้ง่ายว่า หากเหตุการณ์เริ่มมีคุณลักษณะแบบใด ก็อาจมีความเสี่ยงจะกลายเป็นเหตุรุนแรงมากขึ้น

## 🔎 บทสรุป — Cross-Insight Conclusion

จากการวิเคราะห์ข้อมูลสถิติอัคคีภัยปี 2562 ทั้ง 3 ประเด็น สามารถสรุปภาพรวมร่วมกันได้ดังนี้

---

### ภาพรวมที่ค้นพบ

| ประเด็น | สิ่งที่พบ | ความหมาย |
|---|---|---|
| ข้อ 1 — จังหวัดเสี่ยงจริง | ชลบุรี นนทบุรี นครปฐม อุบลราชธานี นครราชสีมา อยู่ในกลุ่ม High Event / High Damage | เป็นจังหวัดที่ควรได้รับทรัพยากรและมาตรการป้องกันก่อน |
| ข้อ 2 — ความเสียหายกระจุกตัว | ภาคตะวันออกเฉียงเหนือมีความเสียหายสะสมสูงสุด โดยเฉพาะสิ่งปลูกสร้างเกษตร | อัคคีภัยในภาคนี้ไม่ได้กระทบแค่ที่อยู่อาศัย แต่กระทบฐานเศรษฐกิจชุมชนโดยตรง |
| ข้อ 3 — ตัวชี้วัดความรุนแรง | fatalities_count มี correlation 0.841 กับมูลค่าความเสียหาย | เหตุการณ์ที่คร่าชีวิตมักเป็น "เหตุใหญ่" ที่เสียหายสูงด้วยเสมอ |

---

### ข้อสรุปเชิงนโยบาย

**1. จัดลำดับความสำคัญเชิงพื้นที่**  
จังหวัดที่อยู่ในกลุ่ม High Event / High Damage ควรได้รับการจัดสรรกำลังพลดับเพลิง งบประมาณป้องกัน
และระบบแจ้งเตือนก่อนจังหวัดอื่น โดยเฉพาะ **ชลบุรี** ซึ่งเป็น hotspot ทั้งด้านจำนวนเหตุ ความเสียหาย
และปรากฏชัดบน Choropleth Map

**2. เฝ้าระวังเหตุที่มีสัญญาณเสี่ยงต่อชีวิต**  
เนื่องจากการสูญเสียชีวิตเป็นตัวทำนายความเสียหายรุนแรงที่แม่นยำที่สุด ระบบรับแจ้งเหตุ
ควรมีโปรโตคอลพิเศษสำหรับเหตุที่มีรายงานผู้ติดอยู่หรือมีผู้บาดเจ็บ เพื่อระดมทรัพยากรให้รวดเร็ว
ก่อนที่ความเสียหายจะบานปลาย

**3. ออกแบบมาตรการให้ตรงกับบริบทพื้นที่**  
ภาคตะวันออกเฉียงเหนือมีความเสียหายสูงในสิ่งปลูกสร้างเกษตรและที่อยู่อาศัยทั้งหลัง
มาตรการที่เหมาะสมจึงควรเน้น **การป้องกันไฟไหม้ในชุมชนเกษตร** เช่น การจัดการวัสดุแห้ง
ช่วงฤดูแล้ง และการฝึกซ้อมดับเพลิงในระดับหมู่บ้าน ซึ่งต่างจากมาตรการในเขตเมืองหรือภาคอุตสาหกรรม

---

### ข้อจำกัดของการศึกษา

- ข้อมูลครอบคลุมเพียง 65 จากทั้งหมด 77 จังหวัด จึงอาจมีพื้นที่เสี่ยงที่ยังไม่ปรากฏในผล
- ข้อมูลบางจังหวัดไม่มีค่ามูลค่าความเสียหาย ทำให้การจัดกลุ่มความเสี่ยงบางกรณีต่ำกว่าความเป็นจริง
- ข้อมูลเป็นของปี 2562 เพียงปีเดียว ยังไม่สามารถวิเคราะห์แนวโน้มระยะยาวได้